# Spain National PV Generation Download — ESIOS

Downloads hourly national solar PV generation from the REE ESIOS API.

**Indicator used:** `541` — *Generación fotovoltaica* (Spain national, all PV)

**Period:** 2023-01-01 → 2025-12-31 (UTC)

**Output:** `spain_total/data/spain_pv_generation.csv`

---
### Prerequisites
You need a free ESIOS API token. Register at https://www.esios.ree.es/en/analysis and request a token from the contact form.
Set it in the cell below or as the environment variable `ESIOS_TOKEN`.

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from datetime import datetime, timezone

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
ESIOS_TOKEN   = os.getenv('ESIOS_TOKEN', 'PASTE_YOUR_TOKEN_HERE')
INDICATOR_ID  = 541          # Generación fotovoltaica — Spain national PV
START_DATE    = '2023-01-01'
END_DATE      = '2025-12-31'
OUTPUT_DIR    = 'data'
OUTPUT_FILE   = os.path.join(OUTPUT_DIR, 'spain_pv_generation.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert ESIOS_TOKEN != 'PASTE_YOUR_TOKEN_HERE', \
    'Set ESIOS_TOKEN env var or paste your token above'
print(f'Token loaded: {ESIOS_TOKEN[:6]}...')
print(f'Indicator:    {INDICATOR_ID}')
print(f'Period:       {START_DATE} -> {END_DATE}')

## 1. Download from ESIOS API

The ESIOS API caps responses at ~1 year, so we download year by year and concatenate.

**Endpoint:** `GET https://api.esios.ree.es/indicators/{id}?start_date=...&end_date=...&time_trunc=hour`

Timestamps are returned in **CET/CEST** (Europe/Madrid) with UTC offsets (+01:00 / +02:00).
We convert to UTC immediately on ingestion.

In [ ]:
BASE_URL = 'https://api.esios.ree.es/indicators'

HEADERS = {
    'Accept':        'application/json; application/vnd.esios-api-v2+json',
    'Content-Type':  'application/json',
    'Authorization': f'Token token="{ESIOS_TOKEN}"',
    'Host':          'api.esios.ree.es',
}

# Year ranges to download (API is reliable year-by-year)
YEAR_RANGES = [
    ('2023-01-01T00:00:00', '2023-12-31T23:59:59'),
    ('2024-01-01T00:00:00', '2024-12-31T23:59:59'),
    ('2025-01-01T00:00:00', '2025-12-31T23:59:59'),
]


def fetch_indicator_year(indicator_id: int, start: str, end: str) -> pd.DataFrame:
    """Fetch one year of hourly data for an ESIOS indicator. Returns UTC-indexed DataFrame."""
    url = f'{BASE_URL}/{indicator_id}'
    params = {
        'start_date': start,
        'end_date':   end,
        'time_trunc': 'hour',
    }
    resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
    resp.raise_for_status()

    data = resp.json()
    values = data['indicator']['values']

    if not values:
        raise ValueError(f'No values returned for {start} → {end}')

    df = pd.DataFrame(values)
    # ESIOS returns 'datetime' in CET/CEST with explicit offset
    df['datetime_utc'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_localize(None)
    df = df[['datetime_utc', 'value']].rename(columns={'value': 'pv_generation_mwh'})
    df = df.sort_values('datetime_utc').reset_index(drop=True)

    return df


print('Starting download...')
frames = []
for start, end in YEAR_RANGES:
    print(f'  Fetching {start[:4]}...', end=' ', flush=True)
    df_year = fetch_indicator_year(INDICATOR_ID, start, end)
    frames.append(df_year)
    print(f'{len(df_year)} rows  [{df_year["datetime_utc"].min()} -> {df_year["datetime_utc"].max()}]')
    time.sleep(1)   # be polite to the API

generation = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows before dedup: {len(generation)}')

## 2. Deduplicate & validate

Year-boundary overlap can produce duplicate timestamps. We keep the first occurrence and verify no gaps in the hourly series.

In [ ]:
# Drop duplicates on timestamp
n_before = len(generation)
generation = generation.drop_duplicates(subset='datetime_utc').reset_index(drop=True)
print(f'Duplicates removed: {n_before - len(generation)}')

generation = generation.sort_values('datetime_utc').reset_index(drop=True)

# Build expected hourly index (UTC, 2023-01-01 00:00 → 2025-12-31 23:00)
expected_idx = pd.date_range('2023-01-01', '2025-12-31 23:00', freq='h', tz=None)
actual_idx   = pd.DatetimeIndex(generation['datetime_utc'])

missing_ts = expected_idx.difference(actual_idx)
extra_ts   = actual_idx.difference(expected_idx)

print(f'Expected rows : {len(expected_idx)}')
print(f'Actual rows   : {len(generation)}')
print(f'Missing timestamps: {len(missing_ts)}')
print(f'Extra timestamps  : {len(extra_ts)}')

if len(missing_ts) > 0:
    print('\nFirst 20 missing:')
    print(missing_ts[:20].tolist())

In [ ]:
# Generation range sanity check
print('=== Generation statistics (MWh, hourly) ===')
print(generation['pv_generation_mwh'].describe())

# Spain national PV capacity ~30 GW in 2025 → max ~30,000 MWh/h at full output
# Typical summer peak ~18,000–22,000 MWh
max_val = generation['pv_generation_mwh'].max()
print(f'\nMax hourly generation: {max_val:,.0f} MWh  ({max_val/1000:.1f} GWh)')

# Check for negative or implausibly large values
n_negative = (generation['pv_generation_mwh'] < 0).sum()
n_over_cap  = (generation['pv_generation_mwh'] > 35_000).sum()
print(f'Negative values       : {n_negative}')
print(f'Values > 35,000 MWh   : {n_over_cap}')

## 3. Set datetime as index and inspect

In [ ]:
generation = generation.set_index('datetime_utc')
generation.index.name = 'datetime_utc'

print(f'Final dataset: {generation.shape}')
print(f'Date range: {generation.index.min()} → {generation.index.max()}')
generation.head(10)

## 4. Monthly aggregation — sanity check

Compare monthly totals against published REE statistics (~8 TWh national solar PV in summer months).

In [ ]:
monthly = (
    generation
    .resample('ME')
    ['pv_generation_mwh']
    .sum()
    .rename('monthly_total_mwh')
    .reset_index()
)
monthly['monthly_total_gwh'] = monthly['monthly_total_mwh'] / 1000
monthly['year']  = monthly['datetime_utc'].dt.year
monthly['month'] = monthly['datetime_utc'].dt.month

# Pivot for easy reading
pivot = monthly.pivot(index='month', columns='year', values='monthly_total_gwh').round(1)
pivot.index = [f'M{m:02d}' for m in pivot.index]
print('Monthly generation totals (GWh):')
print(pivot.to_string())

annual = monthly.groupby('year')['monthly_total_gwh'].sum()
print(f'\nAnnual totals (GWh):')
print(annual.to_string())
# Expected: ~33,000–38,000 GWh/year for Spain national solar PV in 2023–2025

## 5. Export

In [ ]:
generation.to_csv(OUTPUT_FILE)
print(f'Saved: {OUTPUT_FILE}')
print(f'Shape: {generation.shape}')
print(f'Columns: {generation.columns.tolist()}')
print(f'\nPreview:')
generation.head()

## Summary

| Field | Value |
|---|---|
| **Source** | REE ESIOS API |
| **Indicator** | 541 — Generación fotovoltaica (Spain national) |
| **Granularity** | Hourly |
| **Timezone** | UTC (converted from CET/CEST) |
| **Period** | 2023-01-01 → 2025-12-31 |
| **Output** | `spain_total/data/spain_pv_generation.csv` |

**Notes:**
- ESIOS indicator 541 = photovoltaic only (excludes CSP / solar thermal)
- Values are hourly energy in **MWh** (ESIOS native unit)
- This national aggregate will be disaggregated to per-plant targets in notebook `2_data_assembly.ipynb` using pvlib-weighted proportional allocation